# Week 7.2 — Additive Schwarz Preconditioner (Two-Subdomain, 1D)
Domain: indices `0..n-1`; overlap nodes are `i=alpha:beta` (inclusive).

In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve, cg, LinearOperator
import matplotlib.pyplot as plt

## Domain decomposition setup
Splits the 1D grid of $n=127$ points into two overlapping subdomains $I_1=[0,i_L)$ and $I_2=[i_R-1,n)$ with $i_R \le i_L$, so the two pieces share an overlap region of width $i_L-i_R+1$ — the classical two-subdomain, overlapping **Schwarz** setup. `Ao1`/`Ao2` are the corresponding local Poisson submatrices, and `w1`/`w2` form a partition of unity that halves the weight of each subdomain's contribution inside the overlap.

In [2]:
n = 127
h = 1 / (n + 1)
e = np.ones(n)
A = sp.spdiags([-e, 2 * e, -e], [-1, 0, 1], n, n, format='csr') / h**2

# Split indices: [0..iL-1] and [iR-1..n-1], with overlap [iR-1..iL-1]
iL = 80
iR = 50  # choose so that iR <= iL (overlap width ~ iL-iR+1)
I1 = np.arange(iL)        # left subdomain dofs
I2 = np.arange(iR - 1, n)  # right subdomain dofs
Ao1 = A[I1, :][:, I1]
Ao2 = A[I2, :][:, I2]

# Partition of unity (half weights on overlap)
w1 = np.zeros(n)
w2 = np.zeros(n)
w1[I1] = 1
w2[I2] = 1
ov = np.intersect1d(I1, I2)
w1[ov] = 0.5
w2[ov] = 0.5

## Preconditioner: M⁻¹ via subdomain solves
Implements the additive Schwarz operator $M^{-1}v = R_1^T D_1 A_1^{-1}R_1 v + R_2^T D_2 A_2^{-1}R_2 v$: restrict $v$ to each subdomain, solve the small local Poisson problem there (`spsolve`), then prolong back to the full grid and combine with the partition-of-unity weights. Wrapping it in a `LinearOperator` lets it plug directly into `cg`'s `M=` preconditioner argument.

In [3]:
# Apply M^{-1} v = R1^T D1 A1^{-1} R1 v + R2^T D2 A2^{-1} R2 v
def apply_Minv(v):
    sol1 = spsolve(Ao1, v[I1])
    sol2 = spsolve(Ao2, v[I2])
    v1_ext = np.zeros(n)
    v2_ext = np.zeros(n)
    v1_ext[I1] = sol1
    v2_ext[I2] = sol2
    return w1 * v1_ext + w2 * v2_ext

# Build a function-handle preconditioner for pcg
Mfun = LinearOperator((n, n), matvec=apply_Minv)

## Test problem
Another manufactured-solution test, $u_{\text{exact}}=\sin(2\pi x)$, with the matching source term for validation.

In [4]:
xgrid = np.arange(1, n + 1) * h
uex = np.sin(2 * np.pi * xgrid)
f = (2 * np.pi)**2 * uex
b = f

## (a) Stand-alone additive Schwarz iteration `u^{k+1} = u^k + M^{-1}(b - A u^k)`
Runs the Schwarz method as a *stationary* fixed-point iteration on its own (without CG acceleration), analogous to how Jacobi/Gauss-Seidel were used directly in Weeks 1 and 3.

In [5]:
u = np.zeros(n)
rk = b - A @ u
resA = np.linalg.norm(rk)
res_sch = [resA]
itA = 0
while resA > 1e-8 and itA < 200:
    u = u + apply_Minv(rk)
    rk = b - A @ u
    resA = np.linalg.norm(rk)
    res_sch.append(resA)
    itA += 1

## (b) PCG with the same Schwarz preconditioner
Uses the identical operator $M^{-1}$, but now as a *preconditioner* inside CG rather than as a stand-alone iteration — since $A$ is SPD, this is valid as long as $M^{-1}$ is itself SPD, and typically converges much faster than using the Schwarz sweep alone.

In [6]:
tol = 1e-8
maxit = 500
x0 = np.zeros(n)
res_pcg = []
def callback_pcg(xk):
    res_pcg.append(np.linalg.norm(b - A @ xk))

res_pcg.append(np.linalg.norm(b - A @ x0))
_, _ = cg(A, b, x0=x0, rtol=tol, maxiter=maxit, M=Mfun, callback=callback_pcg)

## Compare residuals
Plots both residual histories, illustrating the general pattern that using a good stationary method as a *preconditioner* inside a Krylov method (PCG) usually outperforms using that same method as a stand-alone iteration.

In [7]:
plt.figure()
plt.semilogy(np.arange(len(res_sch)), res_sch, 'o-', label='Additive Schwarz (stand-alone)')
plt.semilogy(np.arange(len(res_pcg)), res_pcg, '*-', label='PCG + Schwarz')
plt.grid(True)
plt.xlabel('Iteration')
plt.ylabel('||r_k||_2')
plt.legend(loc='lower left')
plt.title('Schwarz as iterative method vs as PCG preconditioner')
plt.show()

/var/folders/k6/1w07pxzj0mx129drg82_k3_w0000gp/T/ipykernel_73704/3587306798.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
